# Challenge 2: Full run trees, redaction and export

**EDD: Define. Model upgrade framework, step 2 (continued): extract and sanitise the traces.** A selection is a list of root runs. To judge or reuse a trace you need the whole tree (every LLM and tool span),
and customer data has to be removed before anything leaves LangSmith. You'll fetch a run tree with `client.read_run(..., load_child_runs=True)`, write a `redact()` function, and export the
redacted runs for a pro-code environment (CML or GCP).

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

from lab_state import mark_done

## 1. Fetch one whole run tree

The LangSmith UI downloads one span at a time. `client.read_run(run_id, load_child_runs=True)` returns the run with
`child_runs` populated recursively. This cell picks a successful v1.1 refund run and prints the tree in execution order,
then the tool calls, the final answer, and a failing trace (order `A1005` simulates an upstream outage).

In [ ]:
run = next(r for r in lab_sdk.roots(client)
           if lab_sdk.run_meta(r).get("release") == "v1.1" and "refund" in (r.tags or []) and not r.error)
tree = client.read_run(run.id, load_child_runs=True)


def walk(node, depth=0):
    # Yield (run, depth) for the node and every descendant, ordered by start time.
    yield node, depth
    for child in sorted(node.child_runs or [], key=lambda c: c.start_time):
        yield from walk(child, depth + 1)


for node, depth in walk(tree):
    flag = f"  ERROR: {str(node.error)[:60]}" if node.error else ""
    print("  " * depth + f"{node.name} [{node.run_type}]{flag}")

print("\nTool calls and results")
for node, _ in walk(tree):
    if node.run_type == "tool":
        print(f"  {node.name}({lab_sdk.tool_args(node)}) -> {node.outputs}")
print("Final answer:", (tree.outputs or {}).get("answer"))
print("Trajectory  :", (tree.outputs or {}).get("trajectory"))

failed = next(iter(client.list_runs(project_name=PROD, run_type="tool", error=True)), None)
if failed is None:
    print("\nNo failing tool span found yet. Wait a few seconds and re-run this cell.")
else:
    print("\nA failing trace (errors live on spans):")
    for node, depth in walk(client.read_run(failed.trace_id, load_child_runs=True)):
        flag = f"  ERROR: {str(node.error)[:70]}" if node.error else ""
        print("  " * depth + f"{node.name} [{node.run_type}]{flag}")

## 2. Redaction

Inputs contain customer emails and phone numbers. Two places to redact:

- **At the source**, before anything is sent: `Client(hide_inputs=fn, hide_outputs=fn)` runs your function on
  every payload. Use this for data that must never reach LangSmith.
- **At export**, when you copy data out. That is what you do here.

Write `redact(text)` so it replaces emails with `[EMAIL]` and phone numbers with `[PHONE]`. A reference solution is in the next cell.

In [ ]:
# Reference solution: try writing this yourself first.
import re

def redact(text: str) -> str:
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[EMAIL]", text)
    return re.sub(r"\+?\d[\d\s().-]{7,}\d", "[PHONE]", text)

In [ ]:
assert redact("mail dana@example.com") == "mail [EMAIL]", "redact() did not replace the email address"
assert "[PHONE]" in redact("call +1 415-555-0134"), "redact() did not replace the phone number"


def scrub(obj):
    # Apply redact() to every string inside nested dicts and lists.
    if isinstance(obj, str):
        return redact(obj)
    if isinstance(obj, dict):
        return {k: scrub(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [scrub(v) for v in obj]
    return obj


def to_jsonable(obj):
    return json.loads(json.dumps(obj, default=str))


export = []
for node, depth in walk(tree):
    export.append({"depth": depth, "name": node.name, "run_type": node.run_type, "error": node.error,
                   "inputs": scrub(to_jsonable(node.inputs)), "outputs": scrub(to_jsonable(node.outputs))})

os.makedirs(lab_sdk.EXPORT_DIR, exist_ok=True)
with open(f"{lab_sdk.EXPORT_DIR}/tree_redacted.json", "w") as f:
    json.dump(export, f, indent=2)

assert not lab_sdk.EMAIL.search(json.dumps(export)), "an email address survived redaction"
mark_done("/tmp/.sdk-ch2-redact-ok")
print(f"{len(export)} nodes written to tree_redacted.json, no emails left")

## 3. Export for a pro-code environment

Your team runs evaluation and monitoring in CML or on GCP. The mechanics are the same anywhere Python runs:
pull runs with the SDK, flatten them, write open formats. This cell writes JSONL and Parquet to
`exports/`, using your `redact()` on the way out.

In [ ]:
import time
import pandas as pd

# Fetch again: traces are ingested with a delay, so the list from section 3 can be short.
for _ in range(12):
    all_roots = lab_sdk.roots(client)
    if len(all_roots) >= lab_sdk.EXPECTED_ROOTS:
        break
    print(f"{len(all_roots)} of {lab_sdk.EXPECTED_ROOTS} traces visible yet, waiting...")
    time.sleep(5)

rows = []
for r in all_roots:
    meta = lab_sdk.run_meta(r)
    rows.append({
        "run_id": str(r.id),
        "start_time": r.start_time.isoformat(),
        "status": r.status,
        "error": r.error,
        "tags": r.tags or [],
        "release": meta.get("release"),
        "env": meta.get("env"),
        "thread_id": meta.get("thread_id"),
        "query": redact(str((r.inputs or {}).get("query", ""))),
        "answer": redact(str((r.outputs or {}).get("answer", ""))),
        "trajectory": (r.outputs or {}).get("trajectory"),
        "total_tokens": r.total_tokens,
        "latency_s": (r.end_time - r.start_time).total_seconds() if r.end_time else None,
    })
df = pd.DataFrame(rows)
df.to_json(f"{lab_sdk.EXPORT_DIR}/runs.jsonl", orient="records", lines=True)
df.to_parquet(f"{lab_sdk.EXPORT_DIR}/runs.parquet")
print(len(df), "runs written to", lab_sdk.EXPORT_DIR)
df[["release", "env", "status", "query", "trajectory"]].head()

## 4. See it in LangSmith

In [ ]:
lab_links.project_link(PROD, hint="filter by tag refund, or by metadata release; open a trace to see the tree you just fetched")